


# 1. Monitoring

### Interview definition

**Monitoring is continuously checking an application's health, performance, and behavior using logs and metrics.**

### In very easy words

Monitoring means:

> **“Keep watching the application to know whether it is working properly or something is going wrong.”**

For example, we want to know:

```text
Is the KYC app working?
Is it becoming slow?
Are errors increasing?
Are PASS/REVIEW/FAIL results changing?
```

### In our KYC project

Suppose 1,000 people use your KYC system.

Without monitoring:

```text
Something goes wrong
       ↓
You don't know
       ↓
User complains
       ↓
You investigate manually
```

With monitoring:

```text
Something goes wrong
       ↓
CloudWatch records it
       ↓
You notice the problem
       ↓
You investigate
```

So monitoring answers:

> **“What is happening with my application?”**

---

# 2. Metrics

### Interview definition

**A metric is a numerical measurement used to track the performance or behavior of an application.**

### Easy words

A metric is simply:

> **“A number that tells us something about the system.”**

For example:

```text
Total KYC requests = 100
Average KYC time = 32 seconds
KYC failures = 7
LLM latency = 12 seconds
```

These are all **metrics**.

### In our KYC project

You already created these measurements:

```python
ocr_time
face_time
liveness_time
rag_time
llm_time
total_time
```

These are essentially **performance metrics**.

---

# 3. Logs

### Interview definition

**A log is a record of an event or activity that happened inside an application.**

### Easy words

A log tells us:

> **“What happened?”**

For example, you already have:

```text
INFO - KYC verification started
INFO - OCR completed successfully in 0.625 seconds
INFO - Face verification completed: match passed
INFO - Liveness verification completed with REVIEW status
INFO - LLM KYC assessment completed in 13.917 seconds
```

These are **logs**.

---

# 4. Log vs Metric

This is a very common interview question.

### Easy way to remember

**Log = event/details**

**Metric = number**

Example:

```text
Log:
"LLM KYC assessment completed in 13.917 seconds."

Metric:
LLMLatency = 13.917
```

So:

> **Logs tell us what happened. Metrics help us measure it.**

---

# 5. Why do we need Monitoring in our KYC app?

We don't want to monitor everything.

We want useful information.

For our system, we mainly care about:

### Application health

```text
How many KYC requests?
How many errors?
```

### KYC outcomes

```text
PASS
REVIEW
FAIL
```

### Performance

```text
OCR time
Face time
Liveness time
RAG time
LLM time
Total KYC time
```

This will help us answer questions like:

> Why did KYC become slow?

Maybe:

```text
OCR = 1 sec
Face = 0.7 sec
Liveness = 40 sec  ← problem
RAG = 2 sec
LLM = 12 sec
```

Now we immediately know where the problem is.

---

# 6. What is CloudWatch?

### Interview definition

**Amazon CloudWatch is an AWS monitoring and observability service used to collect, store, visualize, and alert on application logs, metrics, and other system data.**

### Easy words

Think of CloudWatch as:

> **“AWS's monitoring room.”**

Your application sends information to CloudWatch, and CloudWatch helps you see:

```text
What happened?
How often?
How fast?
Are there errors?
Is something getting worse?
```

---

# 7. Our KYC architecture

Right now:

```text
KYC App
   ↓
Python Logging
   ↓
Terminal / Streamlit environment
```

We want to move toward:

```text
KYC App
   ↓
Logs + Metrics
   ↓
CloudWatch
   ↓
Monitoring / Dashboard / Alerts
```

That's the main goal of this topic.

---

# 8. What are we actually going to send?

We should **not** send sensitive KYC information just for monitoring.

We don't need:

```text
Name
DOB
ID Number
Photo
Video
Full prompt
AWS credentials
```

Instead, useful monitoring information could be:

```text
KYCRequests
KYC_PASS
KYC_REVIEW
KYC_FAIL

OCRLatency
FaceLatency
LivenessLatency
RAGLatency
LLMLatency
TotalLatency

KYC_Errors
```

These are enough for our first monitoring implementation.

---

# 9. What will we do first?

Before writing CloudWatch code, we need to understand **two separate things inside CloudWatch**:

### CloudWatch Logs

Stores records such as:

```text
KYC verification started
OCR completed
Face verification passed
```

Think:

> **Logs = detailed history**

### CloudWatch Metrics

Stores numerical values such as:

```text
LLMLatency = 13.9
TotalLatency = 29.0
KYC_FAIL = 1
```

Think:

> **Metrics = numbers we can measure and graph**

That distinction is very important for interviews.

---

## Our learning path from here

We'll go:

**Monitoring → Logs → Metrics → CloudWatch → What our KYC app should send → Implement → Test → Dashboard/alarms**

And we won't jump into code until the concepts are clear.

### One interview line to remember now

> **Monitoring means continuously tracking an application's health and performance using logs and metrics.**

And:

> **A log tells us what happened, while a metric gives us a numerical measurement of what happened.**




## CloudWatch Logs vs CloudWatch Metrics

### 1. CloudWatch Logs

**Interview definition:**
**CloudWatch Logs is an AWS service that stores and lets you monitor log messages generated by applications and services.**

### Easy words

Think:

> **Logs = detailed record of what happened.**

For your KYC app:

```text
KYC verification started
OCR completed in 0.625 seconds
Face verification passed
Liveness status: REVIEW
LLM assessment completed in 13.917 seconds
```

These are events, and we want to keep them as a history.

### Why useful?

Suppose a user says:

> "My KYC failed."

You can look at the logs and see the sequence of events that happened.

---

## 2. CloudWatch Metrics

**Interview definition:**
**CloudWatch Metrics are numerical measurements used to monitor the performance and health of AWS resources and applications.**

### Easy words

Think:

> **Metrics = numbers we measure.**

For our KYC app:

```text
TotalLatency = 29.044 sec
LLMLatency = 13.917 sec
LivenessLatency = 9.023 sec
KYC_PASS = 1
KYC_REVIEW = 1
KYC_FAIL = 0
```

These numbers can be aggregated and displayed as graphs.

---

# 3. Main difference

Remember this:

| Logs                       | Metrics                                  |
| -------------------------- | ---------------------------------------- |
| Detailed events            | Numerical measurements                   |
| Tell you **what happened** | Tell you **how much/how fast/how often** |
| Good for investigation     | Good for trends, graphs, alarms          |

### Example

Your app produces:

```text
Log:
"LLM KYC assessment completed in 13.917 seconds."
```

From that, we can create/record:

```text
Metric:
LLMLatency = 13.917
```

So they can work **together**.

---

# 4. In our KYC system

Our eventual setup is:

```text
                KYC APPLICATION
                       │
              ┌────────┴────────┐
              ↓                 ↓
            LOGS             METRICS
              │                 │
              ↓                 ↓
       CloudWatch Logs   CloudWatch Metrics
              │                 │
              └────────┬────────┘
                       ↓
             Dashboard / Alerts
```

### Why both?

Imagine your LLM suddenly becomes slow.

**Metric tells you:**

> "LLM latency increased from 12 sec to 25 sec."

Then you check **logs** to investigate what happened around that time.

So:

> **Metrics help you detect the problem. Logs help you investigate the problem.**

That's a very good interview explanation.

---

# 5. What we already have in our code

We already generate logs:

```python
logger.info(
    "LLM KYC assessment completed in %.3f seconds",
    llm_time
)
```

And we already calculate the metric-like values:

```python
llm_time
liveness_time
ocr_time
face_time
rag_time
total_time
```

So we're actually **very close**.

We don't need to redesign the KYC pipeline.

We just need to decide:

> **Which of these should become CloudWatch Logs, and which should become CloudWatch Metrics?**

### For our first implementation

**Logs:**

* KYC started/completed
* stage completed
* errors/warnings

**Metrics:**

* Total KYC latency
* LLM latency
* Liveness latency
* OCR latency
* Face latency
* RAG latency
* PASS count
* REVIEW count
* FAIL count
* Error count

---

## Interview memory trick

**Log = story**

**Metric = number**

For example:

> `"Face verification completed: match passed"` → **Log**

> `FaceLatency = 0.671 seconds` → **Metric**




## 3. How does our Python app send data to CloudWatch?

### Interview definition

**An application sends logs and custom metrics to CloudWatch using AWS APIs/SDKs or supported telemetry tools.**

Telemetry in AWS is the automated collection and transmission of data—specifically metrics, logs, and traces—from cloud resources, applications, and IoT devices to monitor and understand system performance

### Easy words

Your Python app produces information:

```text
KYC completed
LLM took 13 seconds
Liveness took 9 seconds
```

Then we **send that information to AWS CloudWatch**.

For Python, we can use **Boto3**, AWS's Python SDK. CloudWatch provides APIs such as `PutMetricData` for publishing custom metrics. ([AWS Documentation][1])

For logs, CloudWatch Logs can receive log events and store them for monitoring and investigation.

---

# 4. Two different paths

Think of it like this:

```text
                    KYC Python App
                         │
             ┌───────────┴───────────┐
             ↓                       ↓
           Logs                    Metrics
             │                       │
             ↓                       ↓
      CloudWatch Logs        CloudWatch Metrics
```

### Logs

We already have:

```python
logger.info(
    "LLM KYC assessment completed in %.3f seconds",
    llm_time
)
```

That produces a log message.

### Metrics

We could explicitly send:

```text
LLMLatency = 13.917
```

to CloudWatch as a custom metric using `PutMetricData`. AWS documents this as a way to publish custom metrics. ([AWS Documentation][3])

---

# 5. What is a Namespace?

This is something you'll see when sending metrics.

### Interview definition

**A namespace is a container used to group related CloudWatch metrics.**

### Easy words

Think of it like a folder.

We could create:

```text
AI-KYC
```

and put our metrics inside it:

```text
AI-KYC
 ├── TotalLatency
 ├── LLM_Latency
 ├── Liveness_Latency
 ├── OCR_Latency
 ├── KYC_PASS
 ├── KYC_REVIEW
 └── KYC_FAIL
```

AWS's `PutMetricData` API uses a namespace, metric name, value, and optionally dimensions when publishing custom metrics. ([AWS Documentation][1])

---

# 6. What is a Dimension?

### Interview definition

**A dimension is a name-value pair that provides additional context about a metric.**

### Easy words

Suppose we have:

```text
TotalLatency = 30 seconds
```

We can add context such as:

```text
Environment = Production
```

Then CloudWatch knows:

> This 30-second measurement belongs to the production environment.

AWS supports dimensions for custom metrics to further identify what the metric represents. ([AWS Documentation][1])

For our first implementation, though, **we don't need to overcomplicate dimensions**.

---

# 7. How this applies to YOUR app

You already calculate:

```python
ocr_time
face_time
liveness_time
rag_time
llm_time
total_time
```

So eventually we'll do something conceptually like:

```text
KYC app
   ↓
calculate llm_time
   ↓
send LLMLatency = llm_time
   ↓
CloudWatch
```

And:

```text
KYC app
   ↓
logger.info(...)
   ↓
CloudWatch Logs
```

So we're **not rebuilding your KYC system**.

We're adding a monitoring layer around the measurements you already created.

---

# 8. One important AWS point

For a **new production implementation**, AWS currently recommends **OpenTelemetry** for publishing custom metrics, although `PutMetricData` is still a supported approach. 

For **our learning project**, we'll first understand and implement the simpler Boto3/CloudWatch approach because it makes the monitoring architecture much easier to understand.

We can then mention OpenTelemetry in your interview as the more modern production option.

### Interview answer

> “For custom application metrics, CloudWatch supports APIs such as PutMetricData. For newer production telemetry, AWS also recommends OpenTelemetry.”

---

## The simple picture to remember

```text
LOG
↓
"What happened?"
↓
CloudWatch Logs


METRIC
↓
"How much / how fast / how often?"
↓
CloudWatch Metrics
```

And:

```text
Python KYC App
      ↓
    Boto3
      ↓
   CloudWatch



## 5. Understanding a CloudWatch Metric


### Interview definition

**A CloudWatch metric is identified by a namespace and metric name, and contains numerical data points that can be monitored over time.**

In easy words:

> **A metric is basically a named number that we keep sending to CloudWatch.**

For example:

```text
Namespace: AI-KYC
Metric Name: TotalLatency
Value: 29.04
Unit: Seconds
```

---

## 1. Namespace

### Interview definition

**A namespace is a container that groups related CloudWatch metrics.**

Easy:

> Think of it as a **folder for your metrics**.

For our project:

```text
AI-KYC
```

Inside it:

```text
AI-KYC
 ├── TotalLatency
 ├── LivenessLatency
 ├── LLMLatency
 ├── KYC_PASS
 ├── KYC_REVIEW
 └── KYC_FAIL
```

---

## 2. Metric Name

### Interview definition

**The metric name identifies what is being measured.**

Easy:

> **What number are we measuring?**

Example:

```text
TotalLatency
```

or

```text
LLMLatency
```

---

## 3. Value

### Interview definition

**The value is the numerical measurement recorded for the metric at a particular time.**

Easy:

> **The actual number.**

Example:

```text
LLMLatency = 13.917
```

Your existing variable:

```python
llm_time
```

can provide that value.

---

## 4. Unit

### Interview definition

**The unit describes what the numerical value represents.**

Easy:

> **What does the number mean?**

For example:

```text
29.04 Seconds
```

For latency metrics:

```text
Unit = Seconds
```

For a count:

```text
KYC_PASS = 1
Unit = Count
```

---

# Putting all four together

For your KYC app:

```text
Namespace:  AI-KYC
Metric:     TotalLatency
Value:      29.04
Unit:       Seconds
```

Meaning:

> In the `AI-KYC` group, the `TotalLatency` measurement was `29.04 seconds`.

Another example:

```text
Namespace:  AI-KYC
Metric:     KYC_REVIEW
Value:      1
Unit:       Count
```

Meaning:

> One KYC case resulted in REVIEW.

---

# 5. What happens when we send it?

Suppose your code finishes a KYC request:

```python
total_time = 29.04
```

We send:

```text
AI-KYC
   ↓
TotalLatency
   ↓
29.04
   ↓
Seconds
```

CloudWatch stores that data point.

Then after many requests, you might have:

```text
Request 1 → 29 sec
Request 2 → 34 sec
Request 3 → 28 sec
Request 4 → 31 sec
Request 5 → 42 sec
```

Now CloudWatch can show the **trend over time**.

That's where monitoring becomes useful.

---

# 6. Why this is better than just a log

You already have:

```text
"KYC verification completed in 29.044 seconds"
```

That's useful for reading.

But with a metric:

```text
TotalLatency = 29.044
```

CloudWatch can treat it as a **number**, which makes it much easier to graph, aggregate, and use for alarms.

So remember:

> **Logs are primarily for detailed investigation. Metrics are primarily for measuring and monitoring trends.**

---

# 7. One more important concept: Data Point

### Interview definition

**A data point is one recorded measurement of a metric at a specific point in time.**

Easy:

> **Every time you send a metric value, you create a data point.**

For example:

```text
10:00 → TotalLatency = 29 sec
10:02 → TotalLatency = 31 sec
10:05 → TotalLatency = 45 sec
```

These are three data points for `TotalLatency`.

---

## Now we're ready for code

We understand:

```text
Namespace → where it belongs
MetricName → what we're measuring
Value → the number
Unit → what the number means
Data point → one measurement at one time
```

### Our first implementation will be deliberately tiny:

We'll send just **one metric** first:

```text
AI-KYC → TotalLatency → total_time → Seconds
```




# Step 1: What is IAM?

### Interview definition

**IAM (Identity and Access Management) is an AWS service used to control who can access AWS resources and what actions they are allowed to perform.**

### Easy words

Think of IAM as the **security/permission system of AWS**.

Suppose your KYC application says:

> "I want to send a metric to CloudWatch."

AWS asks:

> **"Are you allowed to do that?"**

IAM is what answers that question.

---

# 2. What is an IAM User?

### Interview definition

**An IAM user is an identity representing a person or application that can be given AWS permissions.**

### Easy words

Think:

> **IAM User = an AWS identity.**

For example, you might have:

```text
Abhishek
   ↓
IAM User
   ↓
Permissions
```

That user could be allowed to:

```text
Use S3
Use CloudWatch
Read some resources
etc.
```

---

# 3. What is an IAM Role?

This is **very important** for your project.

### Interview definition

**An IAM role is an AWS identity with permissions that can be assumed by an AWS service, application, or trusted entity.**

### Easy words

Think of a role as a **temporary permission badge**.

For example:

```text
Your application
       ↓
Assumes IAM Role
       ↓
Gets permission to use CloudWatch
```

The application doesn't necessarily need a permanent username/password or access key sitting inside the code.

That's why **roles are generally preferred for AWS services**.

---

# 4. User vs Role

The easiest way to remember:

### IAM User

> **Who am I?**

Usually represents a person or long-lived identity.

### IAM Role

> **What permissions does this workload temporarily get?**

Used heavily by AWS services and applications.

Example:

```text
Human:
Abhishek → IAM User

AWS workload:
EC2 → IAM Role
```

---

# 5. What is an IAM Policy?

### Interview definition

**An IAM policy is a JSON document that defines which AWS actions are allowed or denied on which resources.**

### Easy words

Policy = **permission rulebook**.

For example:

```text
Allow:
CloudWatch PutMetricData
```

means:

> "This identity is allowed to send custom metrics to CloudWatch."

---

# 6. Breaking down our policy

We had:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "cloudwatch:PutMetricData"
      ],
      "Resource": "*"
    }
  ]
}
```
In JSON data documents, Sid stands for "Statement ID". It is most commonly used in AWS (Amazon Web Services) IAM policies to name and organize different permission blocks

Let's understand every line.

### `"Effect": "Allow"`

Easy:

> **Allow this action.**

AWS policies can have:

```text
Allow
Deny
```

For us:

```text
Allow
```

---

### `"Action"`

This says:

> **What is the identity allowed to do?**

We have:

```text
cloudwatch:PutMetricData
```

Break that apart:

```text
cloudwatch
    ↓
AWS service

PutMetricData
    ↓
specific operation
```

So:

> **Allow the application to publish custom metrics to CloudWatch.**

---

### `"Resource": "*"`

This means the policy isn't restricting the action to one specific resource ARN.

For `PutMetricData`, AWS permissions are not scoped to a normal resource ARN in the same way many other APIs are, so `*` is used here.

For your interview, the easy understanding is:

> **This permission applies to the CloudWatch metric-publishing operation rather than one specific resource.**

---

# 7. So why do we need IAM at all?

Because AWS follows a basic security principle:

> **No permission → no access.**

Imagine your Python code says:

```python
cloudwatch_client.put_metric_data(...)
```

AWS receives the request.

AWS checks:

```text
Who is making this request?
        ↓
What permissions do they have?
        ↓
Do they have cloudwatch:PutMetricData?
        ↓
YES → allow
NO  → reject
```

That's IAM.

---

# 8. Where does Boto3 fit?

You saw this:

```python
cloudwatch_client = boto3.client(
    "cloudwatch",
    region_name=aws_region
)
```

### Boto3

**Boto3 is the AWS SDK for Python.**

Easy:

> **Boto3 lets Python talk to AWS services.**

So:

```text
Your Python KYC app
        ↓
      Boto3
        ↓
   CloudWatch API
```

But Boto3 alone isn't permission.

Boto3 says:

> "I want to call CloudWatch."

IAM says:

> "Are you allowed to call CloudWatch?"

---

# 9. Where do credentials fit?

This is another important distinction.

AWS needs to know **who is making the request**.

Credentials are one way of proving that identity.

Historically, this can be:

```text
Access Key ID
Secret Access Key
```

That's exactly why we previously had the security issue when credentials were hardcoded inside your code.

### Bad

```python
aws_access_key = "..."
aws_secret_key = "..."
```

Never commit that to GitHub.

### Better

Use an AWS-provided identity mechanism appropriate to where the application runs, such as an IAM role or managed secret/credential mechanism.

---

# 10. For YOUR Streamlit application

Here's where things get a little important.

Your app is currently running through **Streamlit**, and we need to distinguish:

```text
Local VS Code
        ↓
your laptop credentials/environment
```

from:

```text
Streamlit Cloud
        ↓
deployment environment
```

The correct AWS authentication method can differ depending on where the app is running.

So **we should NOT blindly create an IAM user or paste credentials into the code.**

First we should determine **how your current S3 connection is authenticated**.

Because you already have working code that uploads to S3.

---

# 11. The big picture

Memorize this:

```text
IAM
│
├── User
│    └── Identity for a person/long-lived identity
│
├── Role
│    └── Permission identity used by workloads/services
│
└── Policy
     └── Rules saying what that identity can do
```

And:

```text
Python
   ↓
Boto3
   ↓
AWS API
   ↓
IAM checks permissions
   ↓
Allowed / Denied
```

### Interview answer

If an interviewer asks:

**"Why do you need IAM when using CloudWatch?"**

You can say:

> **"IAM controls authentication and authorization for AWS resources. In my KYC application, the application's AWS identity needs permission such as `cloudwatch:PutMetricData` to publish custom monitoring metrics."**




**AWS did not magically know from your project that you wanted CloudWatch.**

It knew because **we explicitly told IAM** through the policy JSON.

The key line was:

```json
"Action": [
    "cloudwatch:PutMetricData"
]
```

AWS reads that as:

> **Service = CloudWatch**
> **Action = PutMetricData**

So the AWS policy editor recognized the service name `cloudwatch` and showed:

> **CloudWatch — Limited: Write**

### Think of it like this

Our JSON:

```text
cloudwatch:PutMetricData
        ↑          ↑
      service     action
```

So:

```text
cloudwatch
   ↓
Which AWS service?

PutMetricData
   ↓
What are we allowed to do?
```

That's why the console displayed **CloudWatch**.

### And this is important for interviews

An IAM policy doesn't say:

> "My application uses CloudWatch."

It says:

> **"This identity is allowed to perform these specific AWS actions."**

For example:

```json
"Action": [
    "s3:PutObject"
]
```

would tell AWS:

> This identity can put objects into S3.

Our:

```json
"Action": [
    "cloudwatch:PutMetricData"
]
```

tells AWS:

> This identity can publish metric data to CloudWatch.

So **IAM doesn't decide what your application wants to use**.
**Your policy grants the application permission to use a specific AWS operation.**



our setup is currently:

IAM User: video-kyc-dev
│
├── AIKYCCloudWatchMetrics
│      └── cloudwatch:PutMetricData
│
└── Group: video-kyc-developer
       └── AmazonS3FullAccess

So our existing S3 access is still there, and we've added the specific CloudWatch metric permission we need.